In [ ]:
!pip install -q ultralytics

import os
import shutil
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

print("Ultralytics:", __import__("ultralytics").__version__)

In [ ]:
# ============================================================
# TEST CONFIGURATION
# ============================================================

MODEL_PATH = Path(
    "/kaggle/working/YOLO11_railway_defect_best.pt"
)

DATASET_ROOT = Path(
    "/kaggle/working/railway_defect_binary"
)

DATA_YAML = DATASET_ROOT / "data.yaml"

RESULTS_ROOT = Path(
    "/kaggle/working/railway_yolo11/test_results"
)

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("Model:", MODEL_PATH)
print("Dataset:", DATASET_ROOT)
print("YAML:", DATA_YAML)

In [ ]:
# ============================================================
# VERIFY TESTING FILES
# ============================================================

print("Model exists:", MODEL_PATH.exists())
print("Dataset exists:", DATASET_ROOT.exists())
print("YAML exists:", DATA_YAML.exists())

TEST_IMAGES = DATASET_ROOT / "test" / "images"

print("Test images:", TEST_IMAGES.exists())

if TEST_IMAGES.exists():
    print(
        "Number of test images:",
        len(list(TEST_IMAGES.glob("*")))
    )

In [ ]:
# ============================================================
# LOAD TRAINED YOLO11 MODEL
# ============================================================

model = YOLO(str(MODEL_PATH))

print("YOLO11 model loaded successfully!")

In [ ]:
# ============================================================
# FINAL TEST-SET EVALUATION
# ============================================================

test_results = model.val(
    data=str(DATA_YAML),

    split="test",

    imgsz=640,
    batch=16,

    device=0,

    plots=True,

    project=str(RESULTS_ROOT),
    name="evaluation"
)

print("Test evaluation completed!")

In [ ]:
# ============================================================
# TEST METRICS
# ============================================================

metrics = test_results.results_dict

precision = metrics.get("metrics/precision(B)", 0)
recall = metrics.get("metrics/recall(B)", 0)
map50 = metrics.get("metrics/mAP50(B)", 0)
map5095 = metrics.get("metrics/mAP50-95(B)", 0)

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0
)

print("=" * 50)
print("FINAL TEST RESULTS")
print("=" * 50)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"mAP@50    : {map50:.4f}")
print(f"mAP@50-95 : {map5095:.4f}")

In [ ]:
# ============================================================
# SAVE FINAL TEST EVALUATION
# ============================================================

test_log = pd.DataFrame({
    "model": ["YOLO11n"],
    "dataset": ["Mendeley Rail Defect"],
    "split": ["test"],
    "images": [310],
    "precision": [precision],
    "recall": [recall],
    "f1_score": [f1],
    "mAP50": [map50],
    "mAP50_95": [map5095]
})

TEST_LOG = RESULTS_ROOT / "test_evaluation.csv"

test_log.to_csv(
    TEST_LOG,
    index=False
)

print("Saved:")
print(TEST_LOG)

In [ ]:
# ============================================================
# TEST SET PREDICTIONS
# ============================================================

prediction_results = model.predict(
    source=str(TEST_IMAGES),

    imgsz=640,

    conf=0.25,

    device=0,

    save=True,
    save_conf=True,

    project=str(RESULTS_ROOT),
    name="predictions"
)

print("Test predictions completed!")

In [ ]:
# ============================================================
# HIGH-CONFIDENCE PREDICTIONS
# ============================================================

confidence_results = model.predict(
    source=str(TEST_IMAGES),

    imgsz=640,

    conf=0.50,

    device=0,

    save=True,
    save_conf=True,

    project=str(RESULTS_ROOT),
    name="predictions_conf50"
)

print("Confidence 0.50 predictions completed!")

In [ ]:
# ============================================================
# COUNT DETECTIONS
# ============================================================

total_images = 0
images_with_detection = 0
images_without_detection = 0
total_detections = 0

for result in prediction_results:

    total_images += 1

    count = len(result.boxes)

    total_detections += count

    if count > 0:
        images_with_detection += 1
    else:
        images_without_detection += 1

print("=" * 50)
print("PREDICTION SUMMARY")
print("=" * 50)

print("Images tested       :", total_images)
print("Images with defects :", images_with_detection)
print("Images without      :", images_without_detection)
print("Total detections    :", total_detections)

In [ ]:
# ============================================================
# SAVE PREDICTION SUMMARY
# ============================================================

prediction_summary = pd.DataFrame({
    "total_images": [total_images],
    "images_with_detection": [images_with_detection],
    "images_without_detection": [images_without_detection],
    "total_detections": [total_detections],
    "confidence_threshold": [0.25]
})

prediction_summary.to_csv(
    RESULTS_ROOT / "prediction_summary.csv",
    index=False
)

print("Prediction summary saved.")

In [ ]:
# ============================================================
# FINAL TESTING SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("RAILGUARD — YOLO11 FINAL TEST REPORT")
print("=" * 60)

print("\nModel: YOLO11n")
print("Task: Binary Railway Defect Detection")

print("\nTEST METRICS")
print("-" * 40)
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"mAP@50    : {map50:.4f}")
print(f"mAP@50-95 : {map5095:.4f}")

print("\nTEST DATA")
print("-" * 40)
print("Images:", total_images)

print("\nMODEL")
print("-" * 40)
print(MODEL_PATH)

print("\nTesting completed successfully.")